# Notebook 03 — RF Pseudopotential and Trap Height

This notebook extends the previous electrostatics workflow by constructing a simple RF-style pseudopotential from the electric field magnitude.

It focuses on:

- solving Laplace's equation for a surface-style electrode layout
- computing the electric field from the potential
- constructing a pseudopotential proportional to $|E|^2$
- visualizing the pseudopotential landscape
- estimating a candidate trap height from the centerline minimum

This is still a lightweight numerical model, but it moves closer to an ion-trap design workflow.


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


## Physical note

For a driven RF trap, a common reduced description uses a pseudopotential proportional to the squared electric field magnitude:

$$
\Phi_{\mathrm{pseudo}} \propto |E|^2
$$

A more complete expression includes charge, mass, and RF angular frequency:

$$
\Phi_{\mathrm{pseudo}} = \frac{q^2 |E|^2}{4 m \Omega^2}
$$

In this notebook we work with a normalized pseudopotential based on $|E|^2$ to study geometry and relative structure.


## Grid and electrode geometry

We reuse the same simple surface-style layout as the earlier notebooks:

- left DC-like electrode: negative voltage
- center RF-like electrode: positive voltage
- right DC-like electrode: negative voltage
- grounded outer boundaries


In [ ]:
# Domain size and resolution
nx, ny = 121, 81
x = np.linspace(-3.0, 3.0, nx)
y = np.linspace(0.0, 4.0, ny)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Potential array and fixed-node mask
V = np.zeros((ny, nx), dtype=float)
fixed = np.zeros((ny, nx), dtype=bool)

# Ground outer boundaries
V[:, 0] = 0.0
V[:, -1] = 0.0
V[-1, :] = 0.0
fixed[:, 0] = True
fixed[:, -1] = True
fixed[-1, :] = True

# Lower boundary electrode pattern
bottom = 0
left_dc = (x >= -2.2) & (x <= -0.9)
center_rf = (x >= -0.5) & (x <= 0.5)
right_dc = (x >= 0.9) & (x <= 2.2)
remaining_ground = ~(left_dc | center_rf | right_dc)

V[bottom, left_dc] = -1.0
V[bottom, center_rf] = 1.0
V[bottom, right_dc] = -1.0
V[bottom, remaining_ground] = 0.0
fixed[bottom, :] = True

print(f'Grid: {nx} x {ny}')
print(f'dx = {dx:.3f}, dy = {dy:.3f}')


## Solve Laplace equation


In [ ]:
def idx(i, j, nx):
    return i * nx + j

N = nx * ny
A = lil_matrix((N, N), dtype=float)
b = np.zeros(N, dtype=float)

cx = 1.0 / dx**2
cy = 1.0 / dy**2
c0 = -2.0 * (cx + cy)

for i in range(ny):
    for j in range(nx):
        k = idx(i, j, nx)

        if fixed[i, j]:
            A[k, k] = 1.0
            b[k] = V[i, j]
        else:
            A[k, idx(i, j, nx)] = c0
            A[k, idx(i, j - 1, nx)] = cx
            A[k, idx(i, j + 1, nx)] = cx
            A[k, idx(i - 1, j, nx)] = cy
            A[k, idx(i + 1, j, nx)] = cy

solution = spsolve(A.tocsr(), b)
Vsol = solution.reshape((ny, nx))

print('Laplace solve complete.')
print(f'Potential range: [{Vsol.min():.3f}, {Vsol.max():.3f}]')


## Compute electric field and pseudopotential


In [ ]:
dV_dy, dV_dx = np.gradient(Vsol, dy, dx)
Ex = -dV_dx
Ey = -dV_dy
E_mag = np.sqrt(Ex**2 + Ey**2)

# Normalized pseudopotential
Phi_pseudo = E_mag**2

print(f'Field magnitude range: [{E_mag.min():.3e}, {E_mag.max():.3e}]')
print(f'Normalized pseudopotential range: [{Phi_pseudo.min():.3e}, {Phi_pseudo.max():.3e}]')


## Potential, field, and pseudopotential plots


In [ ]:
X, Y = np.meshgrid(x, y)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(X, Y, Vsol, shading='auto')
cs = ax.contour(X, Y, Vsol, levels=15)
ax.clabel(cs, inline=True, fontsize=8)
ax.plot(x[left_dc], np.full(np.sum(left_dc), y[0]), linewidth=6)
ax.plot(x[center_rf], np.full(np.sum(center_rf), y[0]), linewidth=6)
ax.plot(x[right_dc], np.full(np.sum(right_dc), y[0]), linewidth=6)
ax.set_title('2D Electrostatic Potential')
ax.set_xlabel('x')
ax.set_ylabel('y')
fig.colorbar(im, ax=ax, label='Potential')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(X, Y, E_mag, shading='auto')
ax.plot(x[left_dc], np.full(np.sum(left_dc), y[0]), linewidth=6)
ax.plot(x[center_rf], np.full(np.sum(center_rf), y[0]), linewidth=6)
ax.plot(x[right_dc], np.full(np.sum(right_dc), y[0]), linewidth=6)
ax.set_title('Electric Field Magnitude')
ax.set_xlabel('x')
ax.set_ylabel('y')
fig.colorbar(im, ax=ax, label='|E|')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(X, Y, Phi_pseudo, shading='auto')
cs = ax.contour(X, Y, Phi_pseudo, levels=15)
ax.clabel(cs, inline=True, fontsize=8)
ax.plot(x[left_dc], np.full(np.sum(left_dc), y[0]), linewidth=6)
ax.plot(x[center_rf], np.full(np.sum(center_rf), y[0]), linewidth=6)
ax.plot(x[right_dc], np.full(np.sum(right_dc), y[0]), linewidth=6)
ax.set_title('Normalized RF-Style Pseudopotential')
ax.set_xlabel('x')
ax.set_ylabel('y')
fig.colorbar(im, ax=ax, label='Pseudo')
plt.tight_layout()
plt.show()


## Centerline pseudopotential and candidate trap height

A simple first diagnostic is the pseudopotential along the vertical centerline $x=0$.

A local minimum away from the surface gives a candidate trapping height in this reduced model.


In [ ]:
j0 = np.argmin(np.abs(x - 0.0))

y_cut = y.copy()
phi_cut = Phi_pseudo[:, j0].copy()

# Exclude the exact boundary from the trap-height search
search_start = 2
search_stop = ny
local_idx = np.argmin(phi_cut[search_start:search_stop])
trap_idx = search_start + local_idx
trap_height = y_cut[trap_idx]
trap_value = phi_cut[trap_idx]

print(f'Candidate trap height (centerline minimum): y = {trap_height:.3f}')
print(f'Centerline pseudopotential minimum: {trap_value:.3e}')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(y_cut, phi_cut, label='Pseudopotential at x=0')
ax.scatter([trap_height], [trap_value], s=60, label='Candidate minimum')
ax.axvline(trap_height, linestyle='--', linewidth=1)
ax.set_title('Centerline RF-Style Pseudopotential')
ax.set_xlabel('y')
ax.set_ylabel('Pseudo')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 2D marker for the candidate trapping height


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(X, Y, Phi_pseudo, shading='auto')
cs = ax.contour(X, Y, Phi_pseudo, levels=15)
ax.clabel(cs, inline=True, fontsize=8)
ax.scatter([0.0], [trap_height], s=70)
ax.plot(x[left_dc], np.full(np.sum(left_dc), y[0]), linewidth=6)
ax.plot(x[center_rf], np.full(np.sum(center_rf), y[0]), linewidth=6)
ax.plot(x[right_dc], np.full(np.sum(right_dc), y[0]), linewidth=6)
ax.set_title('Candidate Trap Height on the Pseudopotential Map')
ax.set_xlabel('x')
ax.set_ylabel('y')
fig.colorbar(im, ax=ax, label='Pseudo')
plt.tight_layout()
plt.show()


## Notes and next steps

This notebook uses a normalized RF-style pseudopotential based on $|E|^2$.

Natural next steps:

- include explicit physical constants $q$, $m$, and $\Omega$
- vary RF amplitude and geometry
- estimate curvature near the candidate minimum
- compare different electrode layouts
- add DC compensation fields
- scan for stable confinement regions across parameter sweeps

That progression will move the workflow toward practical trapped-ion optimization.
